# 02 — Estratégias de Chunking

**Módulo:** EAI_07 — IA Generativa  
**Submódulo:** 03_RAG  
**Ambiente:** `eai07` (Python 3.11)

---

## O que você vai aprender

- Por que o tamanho e formato do chunk afeta a qualidade da busca
- **Chunk fixo** — divisão por número de palavras com overlap
- **Chunk semântico** — divisão por seções do markdown
- **Chunk com enriquecimento** — adicionar sinônimos e contexto
- **Chunk hierárquico** — chunk pequeno para busca + grande para contexto
- Aplicando nos **AGENT_CONTEXT.md** reais do projeto

---

> 💡 No notebook anterior duas queries falharam:  
> `"como o modelo decide qual ferramenta chamar?"` não encontrou `function_calling`  
> `"ajustar uma linha aos pontos"` não encontrou regressão como primeiro resultado  
> O enriquecimento de chunks resolve exatamente isso.

## Por que chunking importa?

```
Chunk muito GRANDE:
  → Embedding dilui o significado (muitos temas num vetor)
  → LLM recebe contexto longo e difuso

Chunk muito PEQUENO:
  → Perde contexto (frase sem contexto é ambígua)
  → Fragmentos sem sentido completo

Chunk IDEAL:
  → Uma ideia por chunk
  → Contexto suficiente para ser autoexplicativo
  → Enriquecido com sinônimos dos termos técnicos
```

## Setup

In [1]:
import sys, os
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

sys.path.append(os.path.abspath('..'))

modelo = SentenceTransformer('all-MiniLM-L6-v2')
print('Modelo carregado!')


def get_embeddings(textos):
    return modelo.encode(textos, normalize_embeddings=True, show_progress_bar=False)


class IndiceVetorial:
    def __init__(self):
        self.documentos = []
        self.metadados  = []
        self.indice     = None

    def adicionar(self, textos, metadados=None):
        embs = get_embeddings(textos).astype(np.float32)
        if self.indice is None:
            self.indice = faiss.IndexFlatIP(embs.shape[1])
        self.indice.add(embs)
        self.documentos.extend(textos)
        self.metadados.extend(metadados or [{} for _ in textos])

    def buscar(self, query, top_k=3, score_minimo=0.3):
        emb_q = get_embeddings([query]).astype(np.float32)
        scores, indices = self.indice.search(emb_q, top_k)
        return [
            {'chunk': self.documentos[i], 'score': float(s), 'meta': self.metadados[i]}
            for s, i in zip(scores[0], indices[0])
            if s >= score_minimo
        ]

    def __repr__(self):
        n = self.indice.ntotal if self.indice else 0
        return f'IndiceVetorial({n} chunks)'


print('Funções carregadas.')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Modelo carregado!
Funções carregadas.


---
## 1. Estratégia A — Chunk fixo

In [2]:
def chunk_fixo(texto: str, tamanho: int = 100, overlap: int = 20) -> list:
    palavras = texto.split()
    chunks, inicio = [], 0
    while inicio < len(palavras):
        fim = min(inicio + tamanho, len(palavras))
        chunks.append(' '.join(palavras[inicio:fim]))
        inicio += tamanho - overlap
    return chunks


texto_exemplo = """
regressao_manual.ipynb implementa regressão linear por mínimos quadrados de forma manual.
Calcula coeficientes a e b: a = [n·Σ(xy) - Σx·Σy] / [n·Σ(x²) - (Σx)²].
Dataset: altura vs peso com 5 pontos. Visualiza reta, resíduos e MSE.
Base para entender gradient descent em deep learning.
""".strip()

chunks = chunk_fixo(texto_exemplo, tamanho=20, overlap=5)
print(f'Texto: {len(texto_exemplo.split())} palavras → {len(chunks)} chunks\n')
for i, c in enumerate(chunks):
    print(f'  [{i+1}] {c}')

Texto: 44 palavras → 3 chunks

  [1] regressao_manual.ipynb implementa regressão linear por mínimos quadrados de forma manual. Calcula coeficientes a e b: a = [n·Σ(xy) - Σx·Σy]
  [2] a = [n·Σ(xy) - Σx·Σy] / [n·Σ(x²) - (Σx)²]. Dataset: altura vs peso com 5 pontos. Visualiza reta, resíduos e
  [3] pontos. Visualiza reta, resíduos e MSE. Base para entender gradient descent em deep learning.


---
## 2. Estratégia B — Chunk semântico (por seções markdown)

In [3]:
def chunk_por_secao(texto: str) -> list:
    chunks = []
    titulo_atual, linhas_atuais = 'Introdução', []

    for linha in texto.split('\n'):
        if linha.startswith('#'):
            if linhas_atuais:
                conteudo = ' '.join(linhas_atuais).strip()
                if conteudo:
                    chunks.append({'titulo': titulo_atual, 'conteudo': conteudo})
            titulo_atual  = linha.lstrip('#').strip()
            linhas_atuais = []
        elif linha.strip():
            linhas_atuais.append(linha.strip())

    if linhas_atuais:
        conteudo = ' '.join(linhas_atuais).strip()
        if conteudo:
            chunks.append({'titulo': titulo_atual, 'conteudo': conteudo})

    return chunks


agent_context_exemplo = """
### regressao_manual.ipynb
Regressão linear por mínimos quadrados sem sklearn.
Fórmulas dos coeficientes a e b. Dataset altura vs peso, 5 pontos.

### transformacoes_lineares.ipynb
Rotação, escala, reflexão e cisalhamento usando matrizes NumPy.

### algebra_linear.ipynb
Autovalores e autovetores com np.linalg.eig. Aplicação em PCA.
"""

chunks_secoes = chunk_por_secao(agent_context_exemplo)
print(f'{len(chunks_secoes)} chunks por seção:\n')
for c in chunks_secoes:
    print(f"  [{c['titulo']}] {c['conteudo'][:80]}")

3 chunks por seção:

  [regressao_manual.ipynb] Regressão linear por mínimos quadrados sem sklearn. Fórmulas dos coeficientes a 
  [transformacoes_lineares.ipynb] Rotação, escala, reflexão e cisalhamento usando matrizes NumPy.
  [algebra_linear.ipynb] Autovalores e autovetores com np.linalg.eig. Aplicação em PCA.


---
## 3. Estratégia C — Chunk com enriquecimento ⭐

Adiciona sinônimos e termos relacionados — resolve os casos onde  
a query usa palavras diferentes do documento.

In [4]:
ENRIQUECIMENTO = {
    'regressão linear'       : 'regressão linear, ajuste de curva, reta de melhor ajuste, mínimos quadrados, ajustar linha',
    'mínimos quadrados'      : 'mínimos quadrados, regressão linear, ajuste de reta, coeficientes a e b',
    'MSE'                    : 'MSE, Mean Squared Error, erro quadrático médio, métrica de avaliação',
    'CNN'                    : 'CNN, rede convolucional, convolutional neural network, redes convolucionais, conv2D',
    'LSTM'                   : 'LSTM, Long Short-Term Memory, células de memória, gates, sequências temporais',
    'deep learning'          : 'deep learning, aprendizado profundo, redes neurais profundas, DL',
    'ANN'                    : 'ANN, rede neural artificial, perceptron, MLP, multilayer perceptron',
    'GRU'                    : 'GRU, Gated Recurrent Unit, células recorrentes, sequências',
    'KNN'                    : 'KNN, K-Nearest Neighbors, vizinhos mais próximos, classificação por distância',
    'Random Forest'          : 'Random Forest, floresta aleatória, ensemble, árvores de decisão',
    'SVM'                    : 'SVM, Support Vector Machine, máquina de vetores de suporte',
    'embeddings'             : 'embeddings, word embeddings, vetores de palavras, representação vetorial',
    'TF-IDF'                 : 'TF-IDF, term frequency, bag of words, BoW, representação de texto',
    'transformers'           : 'transformers, BERT, GPT, attention, mecanismo de atenção, self-attention',
    'autovalores'            : 'autovalores, autovetores, eigenvalues, eigenvectors, PCA',
    'transformações lineares': 'transformações lineares, matrizes, rotação, escala, cisalhamento, reflexão',
    'OpenCV'                 : 'OpenCV, visão computacional, processamento de imagens, detecção',
    'YOLO'                   : 'YOLO, YOLOv5, detecção de objetos, object detection, bounding box',
    'reconhecimento facial'  : 'reconhecimento facial, face recognition, detecção de rostos, identificação',
    'RAG'                    : 'RAG, Retrieval Augmented Generation, recuperação de documentos, busca semântica',
    'function calling'       : 'function calling, tool calling, ferramentas, tools, agentes, modelo decide, chamar função',
    'prompt engineering'     : 'prompt engineering, zero-shot, few-shot, chain-of-thought, CoT, instruções ao modelo',
    'MLflow'                 : 'MLflow, rastreamento de experimentos, experiment tracking, mlops',
    'drift'                  : 'drift, data drift, monitoramento, degradação do modelo',
}


def enriquecer_chunk(texto: str, modulo: str = '', arquivo: str = '') -> str:
    prefixo = f'[{modulo}' + (f' / {arquivo}' if arquivo else '') + '] ' if modulo else ''
    texto_lower = texto.lower()
    sinonimos = [v for k, v in ENRIQUECIMENTO.items() if k.lower() in texto_lower]
    resultado = prefixo + texto
    if sinonimos:
        resultado += ' | ' + '; '.join(sinonimos)
    return resultado


# Demonstra nos casos problemáticos
casos = [
    ('Function calling: permite ao LLM chamar funções Python definidas pelo desenvolvedor.',
     'EAI_07', '03_function_calling.ipynb'),
    ('Regressão linear por mínimos quadrados: calcula coeficientes a e b manualmente.',
     'EAI_01', 'regressao_manual.ipynb'),
]

print('ANTES → DEPOIS do enriquecimento:\n')
for chunk, modulo, arquivo in casos:
    enriquecido = enriquecer_chunk(chunk, modulo, arquivo)
    print(f'  ANTES : {chunk}')
    print(f'  DEPOIS: {enriquecido[:150]}...')
    print()

ANTES → DEPOIS do enriquecimento:

  ANTES : Function calling: permite ao LLM chamar funções Python definidas pelo desenvolvedor.
  DEPOIS: [EAI_07 / 03_function_calling.ipynb] Function calling: permite ao LLM chamar funções Python definidas pelo desenvolvedor. | function calling, tool cal...

  ANTES : Regressão linear por mínimos quadrados: calcula coeficientes a e b manualmente.
  DEPOIS: [EAI_01 / regressao_manual.ipynb] Regressão linear por mínimos quadrados: calcula coeficientes a e b manualmente. | regressão linear, ajuste de curva,...



In [5]:
# Indexa com enriquecimento e testa as queries problemáticas
chunks_originais = [
    'Vetores 2D e 3D: magnitude, normalização e produto escalar implementados manualmente.',
    'Regressão linear por mínimos quadrados: calcula coeficientes a e b manualmente sem sklearn.',
    'Transformações lineares: rotação, escala, reflexão e cisalhamento usando matrizes NumPy.',
    'Autovalores e autovetores com np.linalg.eig — base para PCA e redução de dimensionalidade.',
    'KNN (K-Nearest Neighbors): classificação por distância euclidiana entre vetores de features.',
    'Projeto Diabetes: modelo de classificação usando Random Forest, acurácia de 85%.',
    'Redes neurais: forward pass é uma sequência de multiplicações matriciais W @ x + b.',
    'LSTM para séries temporais: células de memória com gates de entrada, saída e esquecimento.',
    'CNN para classificação de imagens: redes convolucionais, camadas conv2D, MaxPooling, Dense.',
    'Projeto ArtClassifier: CNN, rede convolucional treinada em dataset de obras de arte.',
    'Word embeddings: representação de palavras como vetores em espaço multidimensional.',
    'TF-IDF: frequência de termos ponderada pela raridade no corpus de documentos.',
    'RAG: combina recuperação de documentos relevantes com geração de texto pelo LLM.',
    'Function calling: permite ao LLM chamar funções Python definidas pelo desenvolvedor.',
    'Prompt engineering: técnicas zero-shot, few-shot e chain-of-thought para melhores respostas.',
]

metadados = [
    {'modulo': 'EAI_01', 'arquivo': 'vetores_basicos.ipynb'},
    {'modulo': 'EAI_01', 'arquivo': 'regressao_manual.ipynb'},
    {'modulo': 'EAI_01', 'arquivo': 'transformacoes_lineares.ipynb'},
    {'modulo': 'EAI_01', 'arquivo': 'algebra_linear.ipynb'},
    {'modulo': 'EAI_02', 'arquivo': 'classificacao_KNN.ipynb'},
    {'modulo': 'EAI_02', 'arquivo': 'Projetos/Diabetes'},
    {'modulo': 'EAI_03', 'arquivo': 'conceito_ann.ipynb'},
    {'modulo': 'EAI_03', 'arquivo': 'conceito_lstm.ipynb'},
    {'modulo': 'EAI_03', 'arquivo': 'conceito_cnn.ipynb'},
    {'modulo': 'EAI_03', 'arquivo': 'Projetos/ArtClassifier'},
    {'modulo': 'EAI_04', 'arquivo': 'word_embeddings.ipynb'},
    {'modulo': 'EAI_04', 'arquivo': 'bow_tfidf.ipynb'},
    {'modulo': 'EAI_07', 'arquivo': '03_RAG'},
    {'modulo': 'EAI_07', 'arquivo': '03_function_calling.ipynb'},
    {'modulo': 'EAI_07', 'arquivo': '02_prompt_engineering.ipynb'},
]

chunks_enriquecidos = [
    enriquecer_chunk(c, m['modulo'], m['arquivo'])
    for c, m in zip(chunks_originais, metadados)
]

indice = IndiceVetorial()
indice.adicionar(chunks_enriquecidos, metadados)
print(f'{indice}\n')

queries = [
    'como o modelo decide qual ferramenta chamar?',
    'como ajustar uma linha aos pontos de dados?',
    'qual projeto usou redes convolucionais?',
    'como funcionam as células de memória em sequências?',
    'qual arquivo eu abro para ver regressão linear?',
]

print('Resultados COM enriquecimento:')
print('=' * 65)
for query in queries:
    print(f"\nQuery: '{query}'")
    for r in indice.buscar(query, top_k=2):
        print(f"  [{r['score']:.4f}] {r['meta']}")

IndiceVetorial(15 chunks)

Resultados COM enriquecimento:

Query: 'como o modelo decide qual ferramenta chamar?'
  [0.3516] {'modulo': 'EAI_01', 'arquivo': 'vetores_basicos.ipynb'}

Query: 'como ajustar uma linha aos pontos de dados?'
  [0.3056] {'modulo': 'EAI_04', 'arquivo': 'bow_tfidf.ipynb'}

Query: 'qual projeto usou redes convolucionais?'
  [0.5088] {'modulo': 'EAI_03', 'arquivo': 'Projetos/ArtClassifier'}
  [0.4741] {'modulo': 'EAI_03', 'arquivo': 'conceito_cnn.ipynb'}

Query: 'como funcionam as células de memória em sequências?'
  [0.5892] {'modulo': 'EAI_03', 'arquivo': 'conceito_lstm.ipynb'}
  [0.3664] {'modulo': 'EAI_07', 'arquivo': '03_RAG'}

Query: 'qual arquivo eu abro para ver regressão linear?'
  [0.5833] {'modulo': 'EAI_01', 'arquivo': 'regressao_manual.ipynb'}
  [0.3807] {'modulo': 'EAI_01', 'arquivo': 'transformacoes_lineares.ipynb'}


---
## 4. Estratégia D — Chunk hierárquico

**Chunk pequeno** para busca precisa + **chunk grande** para contexto rico ao LLM.

In [6]:
def processar_agent_context(conteudo: str, modulo: str) -> list:
    """
    Processa um AGENT_CONTEXT.md completo.
    Retorna chunks com:
    - chunk_busca   : para gerar embedding (busca)
    - chunk_contexto: conteúdo completo enviado ao LLM
    """
    secoes = chunk_por_secao(conteudo)
    chunks = []
    for secao in secoes:
        resumo = ' '.join(secao['conteudo'].split()[:40])
        chunk_busca = enriquecer_chunk(
            f"{secao['titulo']}: {resumo}",
            modulo=modulo
        )
        chunks.append({
            'chunk_busca'   : chunk_busca,
            'chunk_contexto': f"[{modulo} — {secao['titulo']}]\n{secao['conteudo']}",
            'titulo'        : secao['titulo'],
            'modulo'        : modulo,
        })
    return chunks


# Demonstra
exemplo = """
### regressao_manual.ipynb
Regressão linear por mínimos quadrados sem sklearn.
Fórmulas: a = [n·Σ(xy) - Σx·Σy] / [n·Σ(x²) - (Σx)²], b = (Σy - a·Σx) / n.
Dataset: altura vs peso, 5 pontos. MSE e resíduos visualizados.

### algebra_linear.ipynb
Autovalores e autovetores com np.linalg.eig. Aplicação em PCA.
Sistemas lineares: np.linalg.solve(A, b).
"""

chunks_hier = processar_agent_context(exemplo, 'EAI_01')
print(f'{len(chunks_hier)} chunks hierárquicos:\n')
for c in chunks_hier:
    print(f"  Título   : {c['titulo']}")
    print(f"  Busca    : {c['chunk_busca'][:90]}...")
    print(f"  Contexto : {len(c['chunk_contexto'].split())} palavras (vai para o LLM)")
    print()

2 chunks hierárquicos:

  Título   : regressao_manual.ipynb
  Busca    : [EAI_01] regressao_manual.ipynb: Regressão linear por mínimos quadrados sem sklearn. Fórmu...
  Contexto : 37 palavras (vai para o LLM)

  Título   : algebra_linear.ipynb
  Busca    : [EAI_01] algebra_linear.ipynb: Autovalores e autovetores com np.linalg.eig. Aplicação em P...
  Contexto : 15 palavras (vai para o LLM)



---
## 5. Aplicando nos AGENT_CONTEXT.md reais do projeto

In [7]:
def encontrar_agent_contexts(pasta_raiz: str) -> list:
    """
    Varre recursivamente e encontra todos os AGENT_CONTEXT.md.
    Retorna lista de (modulo, caminho).
    """
    encontrados = []
    ignorar = {'.git', 'venv', '.venv', '__pycache__', 'node_modules'}

    for raiz, dirs, arquivos in os.walk(pasta_raiz):
        dirs[:] = [d for d in dirs if d not in ignorar and not d.startswith('.')]
        if 'AGENT_CONTEXT.md' in arquivos:
            caminho = os.path.join(raiz, 'AGENT_CONTEXT.md')
            partes  = raiz.replace('\\', '/').split('/')
            modulo  = next((p for p in partes if p.startswith('EAI_')), os.path.basename(raiz))
            encontrados.append((modulo, caminho))

    return sorted(encontrados)


# Aponta para a raiz do projeto — ajuste se necessário
PROJETO_BASE = os.path.abspath('../..')
print(f'Buscando em: {PROJETO_BASE}\n')

arquivos = encontrar_agent_contexts(PROJETO_BASE)
if arquivos:
    print(f'{len(arquivos)} AGENT_CONTEXT.md encontrado(s):')
    for modulo, caminho in arquivos:
        print(f'  [{modulo}] {os.path.relpath(caminho, PROJETO_BASE)}')
else:
    print('Nenhum encontrado. Verifique PROJETO_BASE.')

Buscando em: C:\Users\Jorge Maques\Documents\Especialista_em_AI

26 AGENT_CONTEXT.md encontrado(s):
  [EAI_01_Fundamentos_Matemática_para_IA] EAI_01_Fundamentos_Matemática_para_IA\AGENT_CONTEXT.md
  [EAI_02_Machine_Learning] EAI_02_Machine_Learning\AGENT_CONTEXT.md
  [EAI_02_Machine_Learning] EAI_02_Machine_Learning\Projetos\Desempenho_dos_Alunos\AGENT_CONTEXT.md
  [EAI_02_Machine_Learning] EAI_02_Machine_Learning\Projetos\Deteccao_Fraudes\AGENT_CONTEXT.md
  [EAI_02_Machine_Learning] EAI_02_Machine_Learning\Projetos\Diabetes\AGENT_CONTEXT.md
  [EAI_03_Deep_Learning] EAI_03_Deep_Learning\AGENT_CONTEXT.md
  [EAI_03_Deep_Learning] EAI_03_Deep_Learning\Conceitos\AGENT_CONTEXT.md
  [EAI_03_Deep_Learning] EAI_03_Deep_Learning\Modelos_Base\AGENT_CONTEXT.md
  [EAI_03_Deep_Learning] EAI_03_Deep_Learning\Projetos_Estudos\AGENT_CONTEXT.md
  [EAI_03_Deep_Learning] EAI_03_Deep_Learning\Projetos_Reais\ArtClassifier\AGENT_CONTEXT.md
  [EAI_03_Deep_Learning] EAI_03_Deep_Learning\Projetos_Reais\Previsa

In [8]:
# Processa e indexa todos
todos_chunks = []
for modulo, caminho in arquivos:
    with open(caminho, 'r', encoding='utf-8') as f:
        conteudo = f.read()
    chunks = processar_agent_context(conteudo, modulo)
    todos_chunks.extend(chunks)
    print(f'  {modulo}: {len(chunks)} chunks')

print(f'\nTotal: {len(todos_chunks)} chunks')

  EAI_01_Fundamentos_Matemática_para_IA: 26 chunks
  EAI_02_Machine_Learning: 45 chunks
  EAI_02_Machine_Learning: 83 chunks
  EAI_02_Machine_Learning: 85 chunks
  EAI_02_Machine_Learning: 62 chunks
  EAI_03_Deep_Learning: 32 chunks
  EAI_03_Deep_Learning: 98 chunks
  EAI_03_Deep_Learning: 44 chunks
  EAI_03_Deep_Learning: 60 chunks
  EAI_03_Deep_Learning: 63 chunks
  EAI_03_Deep_Learning: 47 chunks
  EAI_04_NLP_Classico: 69 chunks
  EAI_04_NLP_Classico: 106 chunks
  EAI_04_NLP_Classico: 74 chunks
  EAI_04_NLP_Classico: 71 chunks
  EAI_04_NLP_Classico: 93 chunks
  EAI_05_NLP_com_Transformers: 113 chunks
  EAI_05_NLP_com_Transformers: 82 chunks
  EAI_06_Visao_Computacional: 89 chunks
  EAI_06_Visao_Computacional: 43 chunks
  EAI_06_Visao_Computacional: 51 chunks
  EAI_06_Visao_Computacional: 42 chunks
  EAI_07_AI_Generative: 11 chunks
  EAI_07_AI_Generative: 20 chunks
  EAI_08_MLOps_e_Implantação: 31 chunks
  EAI_08_MLOps_e_Implantação: 13 chunks

Total: 1553 chunks


In [9]:
if todos_chunks:
    print('Indexando...')
    indice_projeto = IndiceVetorial()
    indice_projeto.adicionar(
        [c['chunk_busca'] for c in todos_chunks],
        [{'modulo': c['modulo'], 'titulo': c['titulo']} for c in todos_chunks]
    )
    print(f'{indice_projeto}\n')

    perguntas = [
        'como implementar uma CNN para classificar imagens?',
        'quais métricas foram usadas para avaliar os modelos?',
        'como funciona o mecanismo de atenção?',
        'qual projeto fez reconhecimento facial?',
        'como o modelo decide qual ferramenta chamar?',
        'como fazer regressão linear sem sklearn?',
    ]

    print('Perguntas reais ao projeto:')
    print('=' * 65)
    for pergunta in perguntas:
        print(f'\n👤 {pergunta}')
        for r in indice_projeto.buscar(pergunta, top_k=2):
            print(f"   [{r['score']:.3f}] {r['meta']}")
else:
    print('Sem chunks para indexar. Verifique PROJETO_BASE.')

Indexando...
IndiceVetorial(1553 chunks)

Perguntas reais ao projeto:

👤 como implementar uma CNN para classificar imagens?
   [0.634] {'modulo': 'EAI_03_Deep_Learning', 'titulo': 'Após Completar os 4 Projetos:'}
   [0.616] {'modulo': 'EAI_03_Deep_Learning', 'titulo': 'FAQ TÉCNICO - PROJETOS ESTUDOS'}

👤 quais métricas foram usadas para avaliar os modelos?
   [0.549] {'modulo': 'EAI_01_Fundamentos_Matemática_para_IA', 'titulo': 'MÉTRICAS E RESULTADOS'}
   [0.538] {'modulo': 'EAI_01_Fundamentos_Matemática_para_IA', 'titulo': 'n = 5 pontos'}

👤 como funciona o mecanismo de atenção?
   [0.530] {'modulo': 'EAI_04_NLP_Classico', 'titulo': 'Conceito'}
   [0.529] {'modulo': 'EAI_04_NLP_Classico', 'titulo': 'Estrutura Pedagógica'}

👤 qual projeto fez reconhecimento facial?
   [0.624] {'modulo': 'EAI_06_Visao_Computacional', 'titulo': 'AGENT_CONTEXT.md - Projeto Reconhecimento Facial'}
   [0.476] {'modulo': 'EAI_06_Visao_Computacional', 'titulo': 'RESUMO EXECUTIVO'}

👤 como o modelo decide qual

---
## Resumo

| Estratégia | Quando usar |
|---|---|
| **Fixo** | Texto sem estrutura, PDFs |
| **Semântico** | Markdown com cabeçalhos `###` |
| **Enriquecido** | Termos técnicos com sinônimos |
| **Hierárquico** | `chunk_busca` para embedding + `chunk_contexto` para o LLM |

### Pipeline final

```
AGENT_CONTEXT.md
    ↓ chunk_por_secao()         divide por ###
    ↓ processar_agent_context() chunk_busca + chunk_contexto
    ↓ enriquecer_chunk()        adiciona sinônimos técnicos
    ↓ SentenceTransformer       embeddings 384D normalizados
    ↓ FAISS IndexFlatIP         índice vetorial
Índice pronto para o RAG
```

---